# Feature engineering

In [1]:
%load_ext autoreload
%autoreload 2

import os

os.chdir("..")  # repo root, so `src` imports work

from datetime import timedelta

import polars as pl
from scipy.stats import chatterjeexi, spearmanr
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from tqdm import tqdm

from src.config import CONFIG
from src.data import loader
from src.data.export import CURVES_PATH
from src.data.features import AVAILABLE_LAG_DAYS, LOCAL, build_daily
from src.plotting.charts import heatmap, line_chart
from src.plotting.display import show_df

cfg = CONFIG.model

## When each variable is known

We decide for day D at the auction gate closure, **12:00 CET on D-1**. A variable can only be used once it is published, so each one is shifted by a lag (`AVAILABLE_LAG_DAYS` in `src/data/features.py`):

- **OMIE price ES / PT**: D-1 prices are published around 13:00 on D-2, so all of D-1 is known → **lag 1 day**.
- **OMIE bid curves** (`curva_pbc`): published with the D-1 results → **lag 1 day**.
- **ESIOS `*_d1` forecasts** (demand, wind, solar): issued the day before for day D → **lag 0**.
- **ENTSO-E `load_forecast_da`**: published before 12:00 → **lag 0**.
- **ENTSO-E `transfer_capacity_da`**: day-ahead cross-border capacity, published before the auction → **lag 0**.
- **ENTSO-E `wind_solar_forecast_da`**: deadline is 18:00 D-1, after gate closure → **lag 1 day**.
- **Actuals** (ENTSO-E `load_actual`, `generation_actual`; ESIOS imbalance prices/volume): only D-2 is complete at 12:00 D-1 → **lag 2 days**.
- **Latest forecasts** (ESIOS `demand_forecast`, `wind_forecast`, `solar_pv_forecast`, `residual_demand_forecast`; ENTSO-E `wind_solar_forecast_intraday`): revised up to delivery, so treated like actuals → **lag 2 days**.
- **ENTSO-E `outages`**: event list, not a time series → skipped.

In [2]:
price = loader.load("omie", "marginalpdbc", lazy=True)
esios = {v: loader.load("esios", v, lazy=True) for v in loader.variables("esios")}
entsoe = {v: loader.load("entsoe", v, lazy=True) for v in loader.variables("entsoe") if v != "outages"}

# one long frame: datetime_utc, series, value
long = [price.select("datetime_utc", "price_es", "price_pt").unpivot(index="datetime_utc", variable_name="series")]
long += [lf.select("datetime_utc", pl.lit(name).alias("series"), "value") for name, lf in esios.items()]
for name, lf in entsoe.items():
    # several series per ENTSO-E variable, named "variable|key|key"
    keys = [c for c in ["psr_type", "direction", "out_zone", "in_zone"] if c in lf.collect_schema().names()]
    series = pl.concat_str([pl.lit(name), *[pl.col(k).fill_null("") for k in keys]], separator="|")
    long.append(lf.select("datetime_utc", series.alias("series"), "value"))

hourly = (
    pl.concat(long)
    .with_columns(pl.col("datetime_utc").dt.truncate("1h"))
    .group_by("datetime_utc", "series")
    .agg(pl.col("value").mean().cast(pl.Float64))
    .collect()
    .pivot(on="series", index="datetime_utc", values="value")
    .sort("datetime_utc")
    .upsample("datetime_utc", every="1h")  # missing hours become nulls, so shift(24) is always 24h
)

SERIES = sorted(c for c in hourly.columns if c != "datetime_utc" and (hourly[c].std() or 0) > 0)  # drop flat series
curves = pl.read_parquet(CURVES_PATH).with_columns(pl.exclude("datetime_utc").cast(pl.Float64))
hourly = hourly.select("datetime_utc", *SERIES).join(curves, on="datetime_utc", how="left")
hourly.shape

(67919, 109)

## Targets

One row per delivery day, as in `train()`. Three versions of the battery target:

- `y_spread`: best 4h charge-then-discharge spread after losses (EUR/MWh).
- `y_cycle`: 1 if the spread beats the cycle cost, the label the model predicts.
- `y_spread_change`: spread minus yesterday's, what the naive baseline misses.

In [ ]:
model = build_daily(hourly, start=cfg.start)
FEATURES = [c for c in model.columns if c not in ("day", "y_spread")]
DROPPED = [c for c in SERIES if c not in FEATURES]

other = (
    hourly.with_columns(pl.col(c).shift(24 * AVAILABLE_LAG_DAYS[c.split("|")[0]]) for c in DROPPED)
    .group_by(day=LOCAL.dt.date())
    .agg(pl.col(DROPPED).mean())
)
daily = model.join(other, on="day", how="left").with_columns(
    y_cycle=(pl.col("y_spread") > cfg.cycle_cost).cast(pl.Int8),
    y_spread_change=pl.col("y_spread") - pl.col("spread_1d"),
)
TARGETS = ["y_spread", "y_cycle", "y_spread_change"]

show_df(daily.select(TARGETS).describe(), caption=f"Targets, cycle cost {cfg.cycle_cost:.0f} EUR/MWh")

,statistic,y_spread,y_cycle,y_spread_change
0,count,"2,090.00","2,090.00","2,090.00"
1,null_count,0.00,0.00,0.00
2,mean,49.13,0.77,0.09
3,std,37.61,0.42,31.12
4,min,-18.24,0.00,-161.23
5,25%,21.74,1.00,-15.38
6,50%,39.68,1.00,-0.07
7,75%,72.33,1.00,14.52
8,max,204.42,1.00,179.59


In [4]:
def describe(df: pl.DataFrame) -> pl.DataFrame:
    d = df.describe()
    return d.drop("statistic").transpose(include_header=True, header_name="feature", column_names=d["statistic"])


show_df(
    describe(daily.select(FEATURES + DROPPED)).select(
        "feature",
        pl.when(pl.col("feature").is_in(FEATURES)).then(pl.lit("model")).otherwise(pl.lit("dropped")).alias("used"),
        pl.exclude("feature"),
    ),
    caption="All features, daily means",
)

,feature,used,count,null_count,mean,std,min,25%,50%,75%,max
0,weekday,model,"2,090.00",0.00,4.00,2.00,1.00,2.00,4.00,6.00,7.00
1,month,model,"2,090.00",0.00,6.32,3.39,1.00,3.00,6.00,9.00,12.00
2,holiday,model,"2,090.00",0.00,0.02,0.15,0.00,0.00,0.00,0.00,1.00
3,price_es,model,"2,090.00",0.00,95.62,62.37,0.27,52.20,87.61,124.96,544.98
4,demand_forecast_d1,model,"2,090.00",0.00,"27,310.70","3,003.88","19,299.77","25,329.99","27,242.87","29,406.27","36,269.43"
5,wind_forecast_d1,model,"2,090.00",0.00,"6,764.51","3,647.47",917.83,"3,969.95","5,827.88","8,808.17","19,454.28"
6,solar_pv_forecast_d1,model,"2,090.00",0.00,"4,713.60","2,841.35",308.02,"2,654.75","4,052.79","6,133.54","13,335.30"
7,imbalance_price_up,model,"2,090.00",0.00,71.90,61.26,-132.84,30.82,62.23,92.48,524.02
8,imbalance_price_down,model,"2,090.00",0.00,107.85,65.38,-46.02,66.38,96.36,131.73,559.01
9,generation_actual|B04|in,model,"2,089.00",1.00,"6,462.11","2,860.09","1,969.00","4,335.21","5,745.08","8,098.88","18,166.00"


In [5]:
daily = daily.select("day", *TARGETS, *FEATURES)  # keep only what the model uses
daily.shape

(2090, 47)

## Signal vs the targets

Spearman, Chatterjee xi and mutual information per feature and target, on 180-day windows every 30 days.

In [6]:
WINDOW = timedelta(days=180)
STEP = timedelta(days=30)
MIN_DAYS = 120


def rolling_ic(df: pl.DataFrame, feature: str, target: str) -> pl.DataFrame:
    mi = mutual_info_classif if target == "y_cycle" else mutual_info_regression
    rows = []
    start = df["day"].min()
    while start + WINDOW <= df["day"].max():
        window = (
            df.filter(pl.col("day").is_between(start, start + WINDOW, closed="left"))
            .drop_nulls([feature, target])
            .sample(fraction=1, shuffle=True, seed=0)  # random order for ties, or xi picks up autocorrelation
        )
        # skip short windows and constant ones (y_cycle can be 1 every day)
        if window.height >= MIN_DAYS and window[feature].n_unique() > 1 and window[target].n_unique() > 1:
            x, y = window[feature].to_numpy(), window[target].to_numpy()
            rows.append(
                {
                    "window_start": start,
                    "spearman": spearmanr(x, y).statistic,  # monotonic link, keeps the sign
                    "xi": chatterjeexi(x, y).statistic,  # 0 independent, 1 target is a function of the feature
                    "mi": mi(x.reshape(-1, 1), y, random_state=0)[0],  # any dependence, in nats
                },
            )
        start += STEP
    return pl.DataFrame(rows).with_columns(feature=pl.lit(feature), target=pl.lit(target))


ic = pl.concat(rolling_ic(daily, f, t) for f in tqdm(FEATURES) for t in TARGETS)

summary = (
    ic.group_by("feature", "target")
    .agg(
        spearman=pl.col("spearman").mean(),
        ic_ir=pl.col("spearman").mean() / pl.col("spearman").std(),  # mean IC per unit of noise, > 0.5 is stable
        same_sign=(pl.col("spearman").sign() == pl.col("spearman").mean().sign()).mean(),  # share of windows with the mean's sign
        xi=pl.col("xi").mean(),
        mi=pl.col("mi").mean(),
        windows=pl.len(),
    )
    .sort(pl.col("spearman").abs(), descending=True)
)


  0%|          | 0/43 [00:00<?, ?it/s]


  2%|▏         | 1/43 [00:00<00:31,  1.32it/s]


  5%|▍         | 2/43 [00:01<00:28,  1.43it/s]


  7%|▋         | 3/43 [00:02<00:27,  1.48it/s]


  9%|▉         | 4/43 [00:02<00:25,  1.51it/s]

...


 93%|█████████▎| 40/43 [00:25<00:01,  1.61it/s]


 95%|█████████▌| 41/43 [00:25<00:01,  1.62it/s]


 98%|█████████▊| 42/43 [00:26<00:00,  1.62it/s]


100%|██████████| 43/43 [00:26<00:00,  1.62it/s]


100%|██████████| 43/43 [00:26<00:00,  1.60it/s]

In [7]:
for target in TARGETS:
    top = summary.filter(target=target).drop("target")
    show_df(top.head(10), precision=3, caption=f"Top 10 features for {target}")
    best = top["feature"].head(3).to_list()
    line_chart(
        ic.filter(pl.col("target") == target, pl.col("feature").is_in(best)).pivot(on="feature", index="window_start", values="spearman"),
        "window_start",
        best,
        title=f"Rolling Spearman vs {target}, top 3 features",
    ).show()

heatmap(summary, "target", "feature", "spearman", zmid=0, size="large", title="Mean Spearman per feature and target")

,feature,spearman,ic_ir,same_sign,xi,mi,windows
0,spread_1d,0.428,3.770,1.000,0.133,0.121,64
1,price_range_1d,0.419,3.455,1.000,0.116,0.119,64
2,residual_load,-0.281,-0.788,0.750,0.179,0.226,64
3,import_cover,0.254,0.987,0.828,0.127,0.120,64
4,demand_forecast_d1,-0.234,-0.918,0.766,0.132,0.140,64
5,renewable_share,0.226,0.685,0.688,0.137,0.160,64
6,residual_z_hour_7d,-0.219,-0.629,0.703,0.124,0.144,64
7,price_z_recent_7d,-0.173,-0.750,0.766,0.062,0.090,64
8,price_z_hour_7d,-0.168,-0.756,0.750,0.056,0.080,64
9,demand_z_weekhour_4w,-0.157,-0.672,0.750,0.079,0.086,64


,feature,spearman,ic_ir,same_sign,xi,mi,windows
0,price_range_1d,0.285,2.177,1.000,0.127,0.067,64
1,spread_1d,0.278,2.286,1.000,0.133,0.061,64
2,residual_load,-0.158,-0.549,0.719,0.185,0.076,64
3,import_cover,0.153,0.760,0.750,0.107,0.052,64
4,demand_forecast_d1,-0.137,-0.692,0.734,0.076,0.049,64
5,beta,0.132,0.993,0.873,0.088,0.051,63
6,curve_slope_1gw,0.125,0.671,0.766,0.066,0.028,64
7,curve_implied_change,-0.118,-0.879,0.750,0.050,0.026,64
8,renewable_share,0.118,0.434,0.703,0.126,0.063,64
9,residual_z_hour_7d,-0.107,-0.416,0.641,0.089,0.045,64


,feature,spearman,ic_ir,same_sign,xi,mi,windows
0,spread_1d,-0.508,-8.849,1.000,0.186,0.282,64
1,price_range_1d,-0.465,-6.808,1.000,0.154,0.239,64
2,curve_implied_change,-0.273,-1.190,0.875,0.109,0.115,64
3,curve_implied_price,-0.270,-2.204,0.969,0.069,0.095,64
4,residual_z_hour_7d,-0.250,-2.131,0.969,0.059,0.080,64
5,residual_change_1d,-0.247,-0.870,0.734,0.101,0.140,64
6,weekday,0.214,1.099,0.875,0.060,0.078,64
7,residual_load,-0.209,-2.368,1.000,0.051,0.085,64
8,renewable_share,0.197,2.734,1.000,0.040,0.075,64
9,import_cover,0.180,2.479,0.984,0.022,0.041,64


## Correlation between model features

In [8]:
pairs = [(a, b) for a in FEATURES for b in FEATURES]
rho = daily.select(pl.corr(a, b, method="spearman").alias(str(i)) for i, (a, b) in enumerate(pairs)).row(0)
corr = pl.DataFrame({"x": [a for a, _ in pairs], "y": [b for _, b in pairs], "spearman": rho})

show_df(
    corr.filter(pl.col("x") < pl.col("y"), pl.col("spearman").abs() > 0.8).sort(pl.col("spearman").abs(), descending=True),
    caption="Pairs with |Spearman| > 0.8",
)
heatmap(corr, "x", "y", "spearman", zmid=0, size="large", title="Spearman correlation between model features")

,x,y,spearman
0,price_z_hour_7d,price_z_recent_7d,0.99
1,beta_adjusted_price,price_es,0.94
2,imbalance_price_down,imbalance_price_up,0.94
3,renewable_share,residual_load,-0.92
4,price_range_1d,spread_1d,0.92
5,wind_z_hour_7d,wind_z_recent_2d,0.90
6,imbalance_price_down,price_es,0.90
7,imbalance_price_up,price_es,0.87
8,beta_adjusted_price,imbalance_price_down,0.86
9,wind_forecast_d1,wind_z_hour_7d,0.85
